# 013 - Comparação de arquiteturas MLP

## Objetivo

Partimos da experiência ponderada do 012 e comparamos as arquiteturas `64`, `64 → 32` (controlo), `128 → 64` e `128 → 64 → 32`. Varia apenas a arquitetura. Mantemos ReLU, 16 logits, `CrossEntropyLoss` ponderada pelo inverso da frequência, Adam (`learning_rate=0.001`, `weight_decay=0`), `batch_size=128`, semente 42 e até 100 épocas.

Usamos exatamente a divisão estratificada fixa 80/20 e validação interna do 012. O pré-processador e os pesos de classe são ajustados apenas no treino interno durante a comparação. Cada arquitetura escolhe a primeira época de melhor Macro F1 interna. Selecionamos a melhor arquitetura nova entre as três alternativas e comparamos com o controlo; se o controlo ficar em primeiro, a submissão 013 fica identificada como exploratória.

## Dados e partições

Reutilizamos a preparação comum e repetimos exatamente `test_size=0.2`, `random_state=42` para a divisão fixa, e `test_size=0.15`, `random_state=42`, estratificado, para a validação interna do 012. A validação fixa não participa na comparação nem na escolha de épocas.

In [1]:
import sys
import matplotlib.pyplot as plt
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import torch
from sklearn.model_selection import train_test_split

candidate_roots = [Path.cwd(), *Path.cwd().resolve().parents]
repo_root = next(
    (p for p in candidate_roots if (p / 'notebooks').is_dir() and (p / 'src' / 'preparar_dados.py').is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError('Não foi possível localizar a raiz do projeto com notebooks/ e src/preparar_dados.py.')
sys.path.insert(0, str(repo_root))
from src.preparar_dados import build_preprocessor, make_stratified_split, preparar_dados

train_path = repo_root / 'notebooks' / 'dataset_train.csv'
test_path = repo_root / 'notebooks' / 'dataset_test_X.csv'
if not train_path.is_file() or not test_path.is_file():
    raise FileNotFoundError(f'CSV de treino/teste não encontrados em {repo_root / "notebooks"}')

print(f'Python: {sys.version}')
print(f'PyTorch: {torch.__version__}; scikit-learn: {sklearn.__version__}')
print(f'Dispositivo: CPU')
print(f'Raiz do projeto: {repo_root}')

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
prepared = preparar_dados(train_df, test_df)
X_train, X_val, y_train, y_val = make_stratified_split(
    prepared['X_train'], prepared['y_train'], test_size=0.2, random_state=42
)
assert len(X_train) == 8769 and len(X_val) == 2193
assert sorted(y_train.unique().tolist()) == list(range(16))
print(f'Treino da experiência: {X_train.shape}; validação fixa reservada: {X_val.shape}')

X_inner_train, X_inner_val, y_inner_train, y_inner_val = train_test_split(
    X_train,
    y_train,
    test_size=0.15,
    random_state=42,
    stratify=y_train,
)
assert X_inner_train.columns.equals(X_inner_val.columns)
print(f'Treino interno: {X_inner_train.shape}; validação interna: {X_inner_val.shape}')

Python: 3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, 11:23:37) [Clang 14.0.6 ]
PyTorch: 2.15.0; scikit-learn: 1.6.1
Dispositivo: CPU
Raiz do projeto: /Users/eduardaferreira/daa-kaggle-2026-2027


Treino da experiência: (8769, 236); validação fixa reservada: (2193, 236)
Treino interno: (7453, 236); validação interna: (1316, 236)


## Pré-processamento, arquiteturas e pesos

O pré-processador é ajustado apenas em `X_inner_train`; todas as arquiteturas recebem as mesmas matrizes transformadas. A fórmula de peso é a mesma do 012 (`N / (K × contagem_da_classe)`) e é calculada apenas em `y_inner_train`. O controlo 64 → 32 conserva a arquitetura e o protocolo do 012.

In [2]:
import random
import time
from sklearn.metrics import f1_score

SEED = 42
BATCH_SIZE = 128
MAX_EPOCHS = 100
LEARNING_RATE = 0.001
ARCHITECTURES = {
    '64': (64,),
    '64 → 32': (64, 32),
    '128 → 64': (128, 64),
    '128 → 64 → 32': (128, 64, 32),
}
CONTROL_ARCHITECTURE = '64 → 32'
NUM_CLASSES = 16

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)

def to_dense_float32(matrix):
    if hasattr(matrix, 'toarray'):
        matrix = matrix.toarray()
    return np.asarray(matrix, dtype=np.float32)

inner_preprocessor = build_preprocessor(
    X_inner_train,
    normalize_numeric=True,
)
X_inner_train_array = to_dense_float32(inner_preprocessor.transform(X_inner_train))
X_inner_val_array = to_dense_float32(inner_preprocessor.transform(X_inner_val))
y_inner_train_array = y_inner_train.to_numpy(dtype=np.int64)
y_inner_val_array = y_inner_val.to_numpy(dtype=np.int64)
input_size = X_inner_train_array.shape[1]
estimated_bytes = (X_inner_train_array.nbytes + X_inner_val_array.nbytes) / (1024 ** 2)
print(f'Características transformadas: {input_size}')
print(f'Memória aproximada das matrizes densas: {estimated_bytes:.1f} MiB')

class AgeRangeMLP(torch.nn.Module):
    def __init__(self, input_features, hidden_sizes):
        super().__init__()
        layers = []
        previous_size = input_features
        for hidden_size in hidden_sizes:
            layers.extend([torch.nn.Linear(previous_size, hidden_size), torch.nn.ReLU()])
            previous_size = hidden_size
        layers.append(torch.nn.Linear(previous_size, NUM_CLASSES))
        self.layers = torch.nn.Sequential(*layers)

    def forward(self, features):
        return self.layers(features)

def make_mlp(input_features, hidden_sizes, seed=SEED):
    torch.manual_seed(seed)
    return AgeRangeMLP(input_features, hidden_sizes).to('cpu')

def macro_f1_from_logits(logits, targets):
    predictions = logits.argmax(dim=1).cpu().numpy()
    return f1_score(
        targets,
        predictions,
        labels=list(range(NUM_CLASSES)),
        average='macro',
        zero_division=0,
    )

def evaluate_loss_f1(model, features, targets, criterion):
    model.eval()
    with torch.no_grad():
        logits = model(torch.from_numpy(features))
        loss = criterion(logits, torch.from_numpy(targets))
        score = macro_f1_from_logits(logits, targets)
    return float(loss.item()), float(score)

def inverse_frequency_weights(targets):
    counts = np.bincount(targets, minlength=NUM_CLASSES)
    if np.any(counts == 0):
        raise ValueError(f'Classes ausentes no conjunto usado para calcular pesos: {np.flatnonzero(counts == 0).tolist()}')
    weights = len(targets) / (NUM_CLASSES * counts)
    return torch.tensor(weights, dtype=torch.float32)

print(make_mlp(input_size, ARCHITECTURES[CONTROL_ARCHITECTURE]))
print(f'Pesos inversos calculados apenas no treino interno: {inverse_frequency_weights(y_inner_train_array).tolist()}')

Características transformadas: 2324
Memória aproximada das matrizes densas: 77.7 MiB
AgeRangeMLP(
  (layers): Sequential(
    (0): Linear(in_features=2324, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=16, bias=True)
  )
)
Pesos inversos calculados apenas no treino interno: [3.3036346435546875, 3.9813034534454346, 1.7188652753829956, 1.8125, 1.5272541046142578, 1.0586647987365723, 0.7976241707801819, 0.7949018478393555, 0.757418692111969, 0.6760703921318054, 0.6451696753501892, 0.6588578224182129, 0.6850183606147766, 0.8332960605621338, 1.0538744926452637, 1.378143548965454]


## Treino interno e seleção

Todas as variantes começam com semente 42 e a mesma ordem de minibatches; partilham a mesma perda ponderada e diferem apenas nas camadas ocultas. Treinamos até 100 épocas, guardamos a Macro F1 interna de cada época e selecionamos a primeira época de valor máximo. A melhor arquitetura nova é comparada explicitamente com o controlo.

In [3]:
from torch.utils.data import DataLoader, TensorDataset

class_weights_inner = inverse_frequency_weights(y_inner_train_array)
inner_dataset = TensorDataset(
    torch.from_numpy(X_inner_train_array),
    torch.from_numpy(y_inner_train_array),
)
inner_criterion = torch.nn.CrossEntropyLoss(weight=class_weights_inner)
architecture_histories = {}
selection_rows = []

for architecture_name, hidden_sizes in ARCHITECTURES.items():
    model = make_mlp(input_size, hidden_sizes, seed=SEED)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=0)
    loader_generator = torch.Generator(device='cpu').manual_seed(SEED)
    train_loader = DataLoader(
        inner_dataset, batch_size=BATCH_SIZE, shuffle=True,
        generator=loader_generator, num_workers=0,
    )
    history = {
        'epoch': [], 'train_loss': [], 'validation_loss': [],
        'train_macro_f1': [], 'validation_macro_f1': [],
    }
    start_time = time.perf_counter()
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        for batch_features, batch_targets in train_loader:
            optimizer.zero_grad(set_to_none=True)
            logits = model(batch_features)
            loss = inner_criterion(logits, batch_targets)
            loss.backward()
            optimizer.step()
        train_loss, train_macro_f1 = evaluate_loss_f1(
            model, X_inner_train_array, y_inner_train_array, inner_criterion
        )
        validation_loss, validation_macro_f1 = evaluate_loss_f1(
            model, X_inner_val_array, y_inner_val_array, inner_criterion
        )
        history['epoch'].append(epoch)
        history['train_loss'].append(train_loss)
        history['validation_loss'].append(validation_loss)
        history['train_macro_f1'].append(train_macro_f1)
        history['validation_macro_f1'].append(validation_macro_f1)

    training_seconds = time.perf_counter() - start_time
    best_index = int(np.argmax(history['validation_macro_f1']))
    architecture_histories[architecture_name] = history
    selection_rows.append({
        'architecture': architecture_name,
        'hidden_sizes': hidden_sizes,
        'best_epoch': history['epoch'][best_index],
        'train_macro_f1': history['train_macro_f1'][best_index],
        'validation_macro_f1': history['validation_macro_f1'][best_index],
        'training_seconds': training_seconds,
    })

selection_table = pd.DataFrame(selection_rows)
selection_table = selection_table.sort_values(
    'validation_macro_f1', ascending=False, kind='stable'
).reset_index(drop=True)
display(selection_table)
control_row = selection_table.loc[selection_table['architecture'].eq(CONTROL_ARCHITECTURE)].iloc[0]
new_architectures = selection_table.loc[~selection_table['architecture'].eq(CONTROL_ARCHITECTURE)]
best_new_row = new_architectures.iloc[0]
best_new_architecture = str(best_new_row['architecture'])
control_macro_f1 = float(control_row['validation_macro_f1'])
best_new_macro_f1 = float(best_new_row['validation_macro_f1'])
new_architecture_wins = best_new_macro_f1 > control_macro_f1
if new_architecture_wins:
    selected_architecture = best_new_architecture
    selected_epoch = int(best_new_row['best_epoch'])
else:
    selected_architecture = CONTROL_ARCHITECTURE
    selected_epoch = int(control_row['best_epoch'])
submission_is_exploratory = not new_architecture_wins
print(f'Melhor arquitetura nova: {best_new_architecture} (Macro F1 interna={best_new_macro_f1:.4f})')
print(f'Controlo {CONTROL_ARCHITECTURE}: Macro F1 interna={control_macro_f1:.4f}')
print('A melhor arquitetura nova superou o controlo.' if new_architecture_wins else 'O controlo venceu; a submissão será exploratória.')
print(f'Configuração selecionada para avaliação/submissão: {selected_architecture}; época={selected_epoch}')

,architecture,hidden_sizes,best_epoch,train_macro_f1,validation_macro_f1,training_seconds
0,64 → 32,"(64, 32)",12,0.672711,0.474236,15.517750
1,128 → 64 → 32,"(128, 64, 32)",9,0.612764,0.454312,28.537718
2,64,"(64,)",14,0.793292,0.453672,13.652102
3,128 → 64,"(128, 64)",13,0.773700,0.453220,27.183867


Melhor arquitetura nova: 128 → 64 → 32 (Macro F1 interna=0.4543)
Controlo 64 → 32: Macro F1 interna=0.4742
O controlo venceu; a submissão será exploratória.
Configuração selecionada para avaliação/submissão: 64 → 32; época=12


## Rede nova nos 80% e avaliação única na validação fixa

Reajustamos o pré-processador nos 80% de treino, recalculamos os pesos de classe nesses mesmos exemplos e treinamos uma rede nova da arquitetura selecionada durante o número de épocas escolhido internamente. Depois avaliamos uma única vez nos 20% fixos. Comparamos Macro F1, accuracy e F1 por escalão com o resultado do 012 (Macro F1 **0,4073**), incluindo F1 e recall dos 18–19 anos.

In [4]:
from sklearn.metrics import accuracy_score, classification_report

full_train_preprocessor = build_preprocessor(X_train, normalize_numeric=True)
X_train_array = to_dense_float32(full_train_preprocessor.transform(X_train))
X_val_array = to_dense_float32(full_train_preprocessor.transform(X_val))
y_train_array = y_train.to_numpy(dtype=np.int64)
y_val_array = y_val.to_numpy(dtype=np.int64)
class_weights_full_train = inverse_frequency_weights(y_train_array)
final_criterion = torch.nn.CrossEntropyLoss(weight=class_weights_full_train)

final_model = make_mlp(
    X_train_array.shape[1], ARCHITECTURES[selected_architecture], seed=SEED
)
final_optimizer = torch.optim.Adam(final_model.parameters(), lr=LEARNING_RATE, weight_decay=0)
final_dataset = TensorDataset(torch.from_numpy(X_train_array), torch.from_numpy(y_train_array))
final_generator = torch.Generator(device='cpu').manual_seed(SEED)
final_loader = DataLoader(
    final_dataset, batch_size=BATCH_SIZE, shuffle=True,
    generator=final_generator, num_workers=0,
)
for _ in range(selected_epoch):
    final_model.train()
    for batch_features, batch_targets in final_loader:
        final_optimizer.zero_grad(set_to_none=True)
        logits = final_model(batch_features)
        loss = final_criterion(logits, batch_targets)
        loss.backward()
        final_optimizer.step()

final_model.eval()
with torch.no_grad():
    validation_predictions = final_model(torch.from_numpy(X_val_array)).argmax(dim=1).cpu().numpy()
final_macro_f1 = f1_score(
    y_val_array, validation_predictions,
    labels=list(range(NUM_CLASSES)), average='macro', zero_division=0,
)
final_accuracy = accuracy_score(y_val_array, validation_predictions)
final_report = classification_report(
    y_val_array, validation_predictions,
    labels=list(range(NUM_CLASSES)), target_names=[str(i) for i in range(NUM_CLASSES)],
    output_dict=True, zero_division=0,
)
per_class_metrics = pd.DataFrame([
    {
        'AGE_RANGE': class_id,
        'F1': final_report[str(class_id)]['f1-score'],
        'recall': final_report[str(class_id)]['recall'],
        'precision': final_report[str(class_id)]['precision'],
        'support': int(final_report[str(class_id)]['support']),
    }
    for class_id in range(NUM_CLASSES)
])
print(f'Arquitetura: {selected_architecture}; épocas: {selected_epoch}')
print(f'Macro F1 validação fixa: {final_macro_f1:.4f}')
print(f'Accuracy validação fixa: {final_accuracy:.4f}')
display(per_class_metrics)
class_18_19 = per_class_metrics.loc[per_class_metrics['AGE_RANGE'].eq(1)].iloc[0]
print(f'18–19 anos (classe 1): F1={class_18_19["F1"]:.4f}; recall={class_18_19["recall"]:.4f} ({int(round(class_18_19["recall"] * class_18_19["support"]))}/34)')

Arquitetura: 64 → 32; épocas: 12
Macro F1 validação fixa: 0.4073
Accuracy validação fixa: 0.4168


,AGE_RANGE,F1,recall,precision,support
0,0,0.628571,0.536585,0.758621,41
1,1,0.471698,0.735294,0.347222,34
2,2,0.484076,0.475000,0.493506,80
3,3,0.333333,0.333333,0.333333,75
4,4,0.271605,0.244444,0.305556,90
5,5,0.375000,0.465116,0.314136,129
6,6,0.131148,0.093023,0.222222,172
7,7,0.375335,0.404624,0.350000,173
8,8,0.597826,0.607735,0.588235,181
9,9,0.551320,0.463054,0.681159,203


18–19 anos (classe 1): F1=0.4717; recall=0.7353 (25/34)


## Comparação com o 012

O 012 obteve Macro F1 **0,4073**, accuracy **0,4168** e recall **25/34** nos 18–19 anos. Comparamos também o F1 por escalão, para verificar se uma mudança global esconde regressões em classes específicas.

In [5]:
f1_012 = [
    0.628571, 0.471698, 0.484076, 0.333333,
    0.271605, 0.375000, 0.131148, 0.375335,
    0.597826, 0.551320, 0.573123, 0.377778,
    0.402464, 0.125604, 0.335664, 0.481928,
]
recall_012 = [
    0.536585, 0.735294, 0.475000, 0.333333,
    0.244444, 0.465116, 0.093023, 0.404624,
    0.607735, 0.463054, 0.680751, 0.326923,
    0.490000, 0.078788, 0.369231, 0.606061,
]
class_comparison = per_class_metrics[['AGE_RANGE', 'support', 'F1', 'recall']].copy()
class_comparison['F1_012'] = f1_012
class_comparison['delta_F1'] = class_comparison['F1'] - class_comparison['F1_012']
class_comparison['recall_012'] = recall_012
class_comparison['delta_recall'] = class_comparison['recall'] - class_comparison['recall_012']
display(class_comparison)
metric_comparison = pd.DataFrame([
    {'experiencia': '013 - arquitetura selecionada', 'macro_f1': final_macro_f1, 'accuracy': final_accuracy},
    {'experiencia': '012 - controlo ponderado', 'macro_f1': 0.4073, 'accuracy': 0.4168},
])
metric_comparison['delta_vs_012_macro_f1'] = metric_comparison['macro_f1'] - 0.4073
metric_comparison['delta_vs_012_accuracy'] = metric_comparison['accuracy'] - 0.4168
display(metric_comparison)
print(f'18–19 anos: F1 {class_18_19["F1"]:.4f} vs. 0.4717 no 012; recall {class_18_19["recall"]:.4f} vs. 0.7353 (25/34).')

,AGE_RANGE,support,F1,recall,F1_012,delta_F1,recall_012,delta_recall
0,0,41,0.628571,0.536585,0.628571,4.285714e-07,0.536585,3.658537e-07
1,1,34,0.471698,0.735294,0.471698,1.132075e-07,0.735294,1.176471e-07
2,2,80,0.484076,0.475000,0.484076,4.331210e-07,0.475000,0.000000e+00
3,3,75,0.333333,0.333333,0.333333,3.333333e-07,0.333333,3.333333e-07
4,4,90,0.271605,0.244444,0.271605,-6.172840e-08,0.244444,4.444444e-07
5,5,129,0.375000,0.465116,0.375000,0.000000e+00,0.465116,2.790698e-07
6,6,172,0.131148,0.093023,0.131148,-4.590164e-07,0.093023,2.558140e-07
7,7,173,0.375335,0.404624,0.375335,1.206434e-07,0.404624,2.774566e-07
8,8,181,0.597826,0.607735,0.597826,8.695652e-08,0.607735,-1.933702e-07
9,9,203,0.551320,0.463054,0.551320,-3.519062e-07,0.463054,1.871921e-07


,experiencia,macro_f1,accuracy,delta_vs_012_macro_f1,delta_vs_012_accuracy
0,013 - arquitetura selecionada,0.40728,0.416781,-0.00002,-0.000019
1,012 - controlo ponderado,0.40730,0.416800,0.00000,0.000000


18–19 anos: F1 0.4717 vs. 0.4717 no 012; recall 0.7353 vs. 0.7353 (25/34).


## Conclusão e Decisão
- Data: 10/11
- Conclusão: Os dois CSV têm exatamente as mesmas previsões para os 3.655 RowId, pelo que basta submeter o 012.
- Melhorias:Taxa de aprendizagem. Manter a arquitetura 64 → 32 e a perda ponderada. Comparar 0.0001, 0.001 e 0.01 na mesma validação interna e combinação de modelos. Usar o MLP 012 com o XGBoost 007 e verificar se a mistura melhora a Macro F1 global e o F1 dos 18–19 anos face ao XGBoost sozinho.

- A melhor arquitetura nova foi **128 → 64 → 32** (Macro F1 interna **0,4543**), mas não superou o controlo **64 → 32** (**0,4742**). O controlo foi selecionado para o reajuste final e a submissão 013 fica **exploratória**. Na validação fixa, reproduziu o 012: Macro F1 **0,4073** e accuracy **0,4168**. Para 18–19 anos, o F1 foi **0,4717** e o recall **0,7353 (25/34)**, igual ao 012.

